# Storage 30/60 — value and hedge, at 0 % and 10 %

A seasonal store that fills in 30 days and empties in 60. Two questions only:

1. **What is it worth**, with no funding cost and with gas funded at 10 %?
2. **What do you hedge it with**, in each case?

30/60 divides cleanly — 60 inventory clips, 2 a day in and 1 a day out — so none of the
grid-sizing care that 30/65 needs applies here. `Storage_30_65.ipynb` covers that; this
notebook stays out of it.

In [ ]:
import os, warnings

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import storage_model as sm

pd.set_option("display.width", 200, "display.max_columns", 50)
plt.rcParams.update({"figure.figsize": (12, 3.4), "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})
warnings.filterwarnings("ignore", category=FutureWarning)

if not hasattr(sm, "run_valuation"):
    raise RuntimeError("stale kernel — restart it (Kernel > Restart Kernel and Run All).")

SMOKE = os.environ.get("STORAGE_NOTEBOOK_SMOKE") == "1"
print(f"ready — storage_model from {sm.__file__}")

## 1. The deal

`FUNDING_RATE` is the second case: gas bought in summer has to be paid for and carried until
it is sold in winter, and 10 % a year is the cost of that money. In the model it is
`discount_rate`, applied to every day's cash flow.

The forward curve is **twelve monthly prices, flat within each month** — edit
`MONTHLY_EUR_MWH` below and everything reprices. They repeat each year, which covers the
deal window and the terminal backstop past it.

Flat-per-month is deliberate, and it is not what the contract-curve path does: feeding a
monthly DataFrame through `curve=` runs `smoothen_curve`, which fits a spline so each month's
daily *average* reproduces its contract while the days within it vary. Passing a
`daily_curve` as here uses the series verbatim, so a month is one price — which is how a
monthly-settled gas contract actually pays, and it makes the store's choice of month
unambiguous.

Two things to be clear about. Storage pays on injection and receives on withdrawal, so it has
no single funding *direction* — `borrow_rate`/`invest_rate` are refused for it, and a single
rate is the supported route. And the model settles cash on the day gas moves; a contract
paying later would need its own discount curve.

In [ ]:
WDR_MWH_DAY   = 10_000.0                # maximum withdrawal, MWh/day
INJ_MWH_DAY   = 20_000.0                # maximum injection — twice as fast
CAPACITY      = 600_000.0               # = 60 days out, or 30 days in

START, END    = "2027-01-01", "2027-12-31"
VAL_DATE      = pd.Timestamp("2026-06-01")
VOL, SMR      = 0.50, 1.0
FUNDING_RATE  = 0.10                    # the second case; the first is always 0
N_P           = 12 if SMOKE else 25     # price-tree half-width

N_STATES = int(round(CAPACITY / WDR_MWH_DAY))          # 60 clips of 10,000 MWh
INJ_RATE = int(round(INJ_MWH_DAY / WDR_MWH_DAY))       # 2 clips/day
WDR_RATE = 1                                           # 1 clip/day
assert N_STATES / INJ_RATE == 30.0 and N_STATES / WDR_RATE == 60.0, "grid cannot express 30/60"

print(f"{CAPACITY:,.0f} MWh, {START} .. {END}, valued {VAL_DATE:%Y-%m-%d}")
print(f"  inject   {INJ_MWH_DAY:>8,.0f} MWh/day -> 30 days to fill")
print(f"  withdraw {WDR_MWH_DAY:>8,.0f} MWh/day -> 60 days to empty")
print(f"  grid: {N_STATES} clips of {CAPACITY/N_STATES:,.0f} MWh, "
      f"{INJ_RATE} in / {WDR_RATE} out per day")

# ---- the forward curve: twelve monthly prices, flat within each month --------
# Edit these. They repeat every year, which is all a one-year deal plus its
# terminal backstop needs. The daily curve is a step function -- the model uses a
# `daily_curve` verbatim, so no smoothing is applied and each month trades at one
# price, the way a monthly-settled gas contract does.
MONTHLY_EUR_MWH = {
    "Jan": 30.0, "Feb": 30.0, "Mar": 25.0, "Apr": 25.0,
    "May": 25.0, "Jun": 25.0, "Jul": 25.0, "Aug": 25.0,
    "Sep": 25.0, "Oct": 30.0, "Nov": 30.0, "Dec": 30.0,
}

_MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
           "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
_missing = [m for m in _MONTHS if m not in MONTHLY_EUR_MWH]
if _missing:
    raise ValueError(f"MONTHLY_EUR_MWH is missing {_missing}")

_span = pd.date_range("2026-01-01", "2029-06-30", freq="D")
CURVE = pd.Series([float(MONTHLY_EUR_MWH[_MONTHS[d.month - 1]]) for d in _span], index=_span)

_win = CURVE.loc[START:END]
_tbl = pd.DataFrame({"EUR/MWh": [MONTHLY_EUR_MWH[m] for m in _MONTHS]}, index=_MONTHS).T
display(_tbl.style.format("{:.2f}").set_caption(
    "The input curve — flat within each month, repeating every year"))

fig, ax = plt.subplots()
ax.step(list(_win.index) + [_win.index[-1] + pd.Timedelta(days=1)],
        list(_win.values) + [_win.values[-1]], where="post", color="black", lw=1.4)
ax.axhline(_win.mean(), color="tab:red", ls="--", lw=1,
           label=f"window mean {_win.mean():.2f} EUR/MWh")
ax.fill_between(_win.index, _win.values, _win.mean(), where=_win.values < _win.mean(),
                step="post", color="tab:green", alpha=.18, label="inject below the mean")
ax.fill_between(_win.index, _win.values, _win.mean(), where=_win.values > _win.mean(),
                step="post", color="tab:orange", alpha=.18, label="withdraw above it")
ax.set_ylabel("EUR/MWh"); ax.legend(fontsize=8)
ax.set_title(f"Forward curve, {START} .. {END} — twelve flat months")
plt.tight_layout(); plt.show()

_cheap = min(MONTHLY_EUR_MWH, key=MONTHLY_EUR_MWH.get)
_dear = max(MONTHLY_EUR_MWH, key=MONTHLY_EUR_MWH.get)
print(f"cheapest {_cheap} at {MONTHLY_EUR_MWH[_cheap]:.2f}, dearest {_dear} at "
      f"{MONTHLY_EUR_MWH[_dear]:.2f} — a {max(MONTHLY_EUR_MWH.values()) - min(MONTHLY_EUR_MWH.values()):.2f} "
      f"EUR/MWh spread to store into.")
print(f"window mean {_win.mean():.2f} EUR/MWh over {len(_win)} days.")

## 2. What it is worth

The same deal priced twice. `intrinsic` is the schedule you could fix today against the
forward curve; `extrinsic` is what re-optimising as prices move adds. Both are present values
at the valuation date, and per MWh of **capacity** — storage cycles, so value per net MWh
moved is undefined and the model refuses to report it.

In [ ]:
def price(rate, n_p=None, run_intrinsic=True):
    model, result = sm.run_valuation(None, dict(
        product_type="storage", valDate=VAL_DATE, storageStart=START, storageEnd=END,
        capacity_mwh=CAPACITY, daily_max=INJ_MWH_DAY, clips_per_day=INJ_RATE,
        inj_rate=INJ_RATE, wdr_rate=WDR_RATE,
        initial_inv_clips=0, terminal_inv_clips=0, inj_cost=0.0, wdr_cost=0.0,
        vol=VOL, sMR=SMR, n_p_full=N_P if n_p is None else n_p,
        run_intrinsic=run_intrinsic, discount_rate=rate, daily_curve=CURVE))
    n = model.n_t
    moved = model.prob[:n] * model.strat[:n] * model.v_step
    injected = np.clip(moved, 0, None).sum(axis=(1, 2))
    withdrawn = -np.clip(moved, None, 0).sum(axis=(1, 2))
    value = float(model.v[0, model.n_p, model.initial_state])
    # sum(DF * delta * F) reprices the deal; there is no cost leg here, both costs being 0.
    reprice = float(np.dot(model.d_curve[:n] * np.asarray(model.delta[:n]),
                           np.asarray(model.fwd)[:n]))
    return model, result, dict(
        value=value, injected=injected, withdrawn=withdrawn,
        dates=pd.DatetimeIndex(model.date_span)[:n],
        invariant=abs(reprice - value) / max(abs(value), 1.0))


RUNS = {rate: price(rate) for rate in (0.0, FUNDING_RATE)}

_rows = []
for _rate, (_m, _r, _d) in RUNS.items():
    _rows.append({"funding rate": _rate, "value EUR": _d["value"],
                  "EUR/MWh capacity": _d["value"] / CAPACITY,
                  "intrinsic": _r["intrinsic"], "extrinsic": _r["extrinsic"],
                  "MWh cycled": _d["injected"].sum(),
                  "turns": _d["injected"].sum() / CAPACITY,
                  "invariant": _d["invariant"]})
_val = pd.DataFrame(_rows)
display(_val.style.hide(axis="index").format({
    "funding rate": "{:.0%}", "value EUR": "{:,.0f}", "EUR/MWh capacity": "{:.4f}",
    "intrinsic": "{:.4f}", "extrinsic": "{:.4f}", "MWh cycled": "{:,.0f}",
    "turns": "{:.2f}", "invariant": "{:.0e}"})
    .set_caption(f"30/60 storage, {CAPACITY:,.0f} MWh, vol {VOL:.0%}, "
                 f"valued {VAL_DATE:%Y-%m-%d} — present values"))

_free, _funded = _val.iloc[0], _val.iloc[1]
print(f"funding at {FUNDING_RATE:.0%} costs "
      f"{_free['value EUR'] - _funded['value EUR']:,.0f} EUR, "
      f"{1 - _funded['value EUR']/_free['value EUR']:.1%} of the value.")

# How hard does funding bite? The deterministic run isolates it from optionality.
# Volume is NOT the measure to use on a flat-month curve -- see the note below --
# so this reports the value, and the nominal cash the schedule moves.
_rates = [0.0, FUNDING_RATE] if SMOKE else [0.0, 0.05, 0.10, 0.25, 0.50, 1.00, 2.00]
_rows = []
for _r in _rates:
    _m, _, _d = price(_r, n_p=0, run_intrinsic=False)
    _n = _m.n_t
    _f = np.asarray(_m.price_curve, dtype=float)[:_n]
    _rows.append({"funding rate": _r, "PV EUR": _d["value"],
                  "nominal cash EUR": float(np.dot(_d["withdrawn"], _f)
                                            - np.dot(_d["injected"], _f)),
                  "MWh cycled": _d["injected"].sum()})
_core = pd.DataFrame(_rows)
_core["PV vs 0 %"] = _core["PV EUR"] / _core.loc[0, "PV EUR"] - 1.0
display(_core.style.hide(axis="index").format({
    "funding rate": "{:.0%}", "PV EUR": "{:,.0f}", "nominal cash EUR": "{:,.0f}",
    "MWh cycled": "{:,.0f}", "PV vs 0 %": "{:+.1%}"})
    .set_caption("Deterministic value against funding cost — the schedule's nominal cash "
                 "barely moves, so this is discounting, not a different strategy"))

_carry = _win.min() * FUNDING_RATE * 5 / 12
print(f"carry on summer gas at {FUNDING_RATE:.0%} held five months is about "
      f"{_carry:.2f} EUR/MWh, against a "
      f"{max(MONTHLY_EUR_MWH.values()) - min(MONTHLY_EUR_MWH.values()):.2f} EUR/MWh spread,")
print(f"so the seasonal round trip stays firmly in the money and funding acts as a haircut")
print(f"on an almost unchanged schedule rather than changing what the store does.")
print()
_v0, _v1 = _core.loc[0, "MWh cycled"], _core.iloc[1]["MWh cycled"]
if abs(_v1 - _v0) > 1e-6 * max(_v0, 1.0):
    print(f"Careful with the MWh column. On a curve flat within a month, injecting and")
    print(f"withdrawing inside the same month is EXACTLY break-even, so the optimiser is")
    print(f"indifferent and any non-zero rate tips it into doing it: cycled volume goes")
    print(f"{_v0:,.0f} -> {_v1:,.0f} MWh while the nominal cash is unchanged at "
          f"{_core.loc[0, 'nominal cash EUR']:,.0f} EUR.")
    print(f"That churn is free and worth nothing. Measure the store by its cash, not by how")
    print(f"much gas it moves.")
else:
    print(f"On this curve the volume is stable across rates. Be careful with the MWh column")
    print(f"generally though: where a month is flat, injecting and withdrawing inside it is")
    print(f"exactly break-even, so the optimiser is indifferent and a small rate change can")
    print(f"move the volume a long way without moving the cash at all.")

## 3. What it does

Expected inventory, and the daily movement behind it. Injection runs at twice the withdrawal
rate, so the store fills in a third of the time it takes to empty — the asymmetry is visible
as the slope.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 5.2), sharex=True)
for (_rate, (_m, _r, _d)), _c in zip(RUNS.items(), ("tab:blue", "tab:purple")):
    _inv = np.cumsum(_d["injected"] - _d["withdrawn"])
    ax[0].plot(_d["dates"], _inv, color=_c, lw=1.4, label=f"funding {_rate:.0%}")
ax[0].axhline(CAPACITY, color="tab:red", ls="--", lw=1, label=f"capacity {CAPACITY:,.0f}")
ax[0].set_ylabel("MWh in store"); ax[0].legend(fontsize=8)
ax[0].set_title("Expected inventory — fills in 30 days, empties in 60")

_d0 = RUNS[0.0][2]
ax[1].bar(_d0["dates"], _d0["injected"], width=1.0, color="tab:green", label="injected")
ax[1].bar(_d0["dates"], -_d0["withdrawn"], width=1.0, color="tab:orange", label="withdrawn")
ax[1].axhline(INJ_MWH_DAY, color="tab:green", ls="--", lw=.9)
ax[1].axhline(-WDR_MWH_DAY, color="tab:orange", ls="--", lw=.9)
ax[1].axhline(0, color="black", lw=.8)
ax[1].set_ylabel("MWh/day, at 0 %"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

for _rate, (_m, _r, _d) in RUNS.items():
    _inv = np.cumsum(_d["injected"] - _d["withdrawn"])
    print(f"  {_rate:>4.0%}: peak inventory {_inv.max():>9,.0f} MWh "
          f"({_inv.max()/CAPACITY:5.1%} full), ends at {_inv[-1]:,.0f}, "
          f"peak rates {_d['injected'].max():,.0f} in / {_d['withdrawn'].max():,.0f} out")

## 4. The hedge

`delta` is the MWh of each month's forward you would trade to hedge that month's price risk —
negative to buy, positive to sell. It is **not** the gas: `physical` is what actually moves.

For a store the difference is stark. Physical nets to **zero** over the deal, because
everything injected is withdrawn. The hedge does not, because you buy summer forwards and
sell winter ones at different prices.

The hedge splits the same way the value does:

```
total delta  =  intrinsic delta  +  extrinsic delta
```

**`intrinsic delta` is the deterministic schedule's own volume.** With no price uncertainty
`E[S | exercise] = F`, so the hedge and the schedule coincide exactly — it is what you would
trade today to lock the intrinsic value and then leave alone. **`extrinsic delta` is the
rest**: the part that exists only because prices move, and the part you re-hedge as they do.

`delta_pv` is the total tailed by the discount factor, for use against **margined futures**,
where variation margin moves today while the gas settles at delivery.

In [ ]:
def hedge(rate):
    """Monthly hedge, split into the locked-in part and the part that is not."""
    _m, _, _ = RUNS[rate]
    _det, _, _ = price(rate, n_p=0, run_intrinsic=False)   # no uncertainty -> intrinsic
    n = _m.n_t
    frame = pd.DataFrame({
        "physical": np.asarray(_m.exp_ex[:n]),
        "intrinsic delta": np.asarray(_det.delta[:n]),
        "total delta": np.asarray(_m.delta[:n]),
        "delta_pv": np.asarray(_m.delta_pv[:n])},
        index=pd.DatetimeIndex(_m.date_span)[:n]).loc[START:END].resample("MS").sum()
    frame["extrinsic delta"] = frame["total delta"] - frame["intrinsic delta"]
    frame = frame[["physical", "intrinsic delta", "extrinsic delta", "total delta", "delta_pv"]]
    frame.index = [f"{d:%b}" for d in frame.index]
    frame.loc["TOTAL"] = frame.sum()
    return frame


for _rate in (0.0, FUNDING_RATE):
    _h = hedge(_rate)
    _cols = ["physical", "intrinsic delta", "extrinsic delta", "total delta"]
    if _rate:
        _cols.append("delta_pv")
    display(_h[_cols].style.format("{:,.0f}")
            .set_properties(subset=pd.IndexSlice["TOTAL", :], **{"font-weight": "bold"})
            .set_caption(f"Monthly hedge at {_rate:.0%} funding, MWh of forward — "
                         f"negative buys, positive sells"))

_h = hedge(FUNDING_RATE)
fig, ax = plt.subplots()
_x = np.arange(len(_h) - 1)
ax.bar(_x, _h["intrinsic delta"][:-1], 0.62, color="tab:blue", label="intrinsic delta")
ax.bar(_x, _h["extrinsic delta"][:-1], 0.62, bottom=_h["intrinsic delta"][:-1],
       color="tab:orange", label="extrinsic delta")
ax.axhline(0, color="black", lw=.8)
ax.set_xticks(_x); ax.set_xticklabels(_h.index[:-1])
ax.set_ylabel("MWh of forward"); ax.legend(fontsize=8)
ax.set_title(f"The hedge at {FUNDING_RATE:.0%}: what you can lock in, and what you cannot")
plt.tight_layout(); plt.show()

_i, _e = _h.loc["TOTAL", "intrinsic delta"], _h.loc["TOTAL", "extrinsic delta"]
print(f"physical nets to {_h.loc['TOTAL', 'physical']:,.0f} MWh — a store gives back "
      f"everything it takes.")
print(f"intrinsic delta nets to {_i:,.0f} too: it IS the deterministic schedule, and that")
print(f"schedule is a closed cycle. The whole {_e:,.0f} MWh of net hedge is extrinsic.")

# Which months carry the story depends on the curve, so read them off rather than
# naming them -- MONTHLY_EUR_MWH is an input and these move when it changes.
_body = _h.drop(index="TOTAL")
_locked = _body["intrinsic delta"].idxmin()          # the locked-in purchase month
_held = _body.loc[_locked, "extrinsic delta"]
_elsewhere = _body["extrinsic delta"].idxmin()       # where the option buys instead
print()
if _body.loc[_locked, "intrinsic delta"] < 0:
    print(f"{_locked} carries the locked-in purchase: intrinsic delta "
          f"{_body.loc[_locked, 'intrinsic delta']:,.0f} MWh against a total of "
          f"{_body.loc[_locked, 'total delta']:,.0f}.")
    print(f"The {_held:+,.0f} MWh of extrinsic delta there is the part you do not commit — "
          f"the option")
    print(f"to buy elsewhere instead, and {_elsewhere} is where the largest opposing "
          f"extrinsic position sits at {_body.loc[_elsewhere, 'extrinsic delta']:,.0f} MWh.")
print()
print(f"tailed for margined futures the book is "
      f"{_h.loc['TOTAL', 'delta_pv']:,.0f} MWh instead of {_h.loc['TOTAL', 'total delta']:,.0f}.")
for _rate, (_m, _r, _d) in RUNS.items():
    print(f"  repricing check at {_rate:.0%}: sum(DF x delta x F) matches the value "
          f"to {_d['invariant']:.0e}")

## Traps

- **The intrinsic hedge is the intrinsic schedule.** With no price uncertainty
  `E[S | exercise] = F`, so `delta` collapses onto volume — they agree to 4e-12 here.
  That is why the intrinsic delta nets to zero and every euro of net hedge is
  extrinsic.
- **`physical` and `delta` answer different questions.** How much gas, versus how much price
  risk. For a store they are barely related: physical nets to zero, the hedge does not.
- **Funding is not a haircut.** At 10 % the store both earns less *and* cycles less, because
  the marginal round trip stops covering the carry. §2 shows the volume falling with the rate.
- **Value is per MWh of capacity here**, stated on the table. Per *net* MWh moved is
  undefined for a cycling deal, which is why `profiled()` raises rather than returning zero.
- **One rate, one direction.** Storage pays and receives, so `borrow_rate`/`invest_rate` are
  refused; a single `discount_rate` or an explicit `d_curve` is the route.
- **Flat months make some churn free.** Inject and withdraw inside one month and the
  price is identical, so the round trip is exactly break-even and the optimiser is
  indifferent — any non-zero rate tips it into doing it. Cycled volume doubles while
  nominal cash does not move. Judge the store by cash, not by MWh moved. A smoothed
  curve has a gradient within each month and does not do this.
- **Rates are maximums.** 30/60 caps the daily volume; it does not mean 30 injection days.